# Preprocess SSP Population and GDP Inputs

Resamples SSP population and GDP files from their native high resolution (~0.008°, ~1 km) to 0.10° and applies log1p z-score normalization, matching the preprocessing used for the training data (`fix_data_pop_gdp.ipynb`).

Run this once before `projections.ipynb`. Output goes to `READY_data/inputs_normalized/ssp/`.

In [ ]:
import os
import json
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from rasterio.warp import reproject, calculate_default_transform


## 1. Setup load reference grid

Loads the CISI label raster to extract the target 0.10° grid (extent, CRS, affine transform). All SSP inputs will be reprojected onto this exact grid so every pixel aligns with the training labels.

In [ ]:
OUTPUT_DIR   = r'READY_data\inputs_normalized\ssp'
REFERENCE    = r'READY_data\labels\2024_CISI_010deg_nearest.tif'  # defines the 0.10° grid
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load reference grid
with rasterio.open(REFERENCE) as ref:
    ref_crs       = ref.crs
    ref_transform = ref.transform
    ref_height    = ref.height
    ref_width     = ref.width
    ref_profile   = ref.profile.copy()

print(f'Reference grid: {ref_height}x{ref_width}, res={ref_transform[0]:.3f}°')

Reference grid: 485x570, res=0.100°


## 2. Training-stats normalization + resample function

Normalizes SSP files using **the same log1p mean and std as the training inputs** (2019 GDP, 2020 POP).

This ensures SSP inputs fall in exactly the z-score range the model learned from, while still preserving between-scenario scale differences: SSP5 2100 GDP > SSP1 2030 GDP in absolute terms, so after subtracting the same training mean and dividing by the same training std, SSP5 still has higher z-scores.

**No retraining needed.**

Per-file normalization (old approach) collapsed these differences by independently rescaling each SSP file to mean=0, std=1.

Stats are saved to `NORM_STATS_SSP.json` for reproducibility and verification.

In [ ]:
RESAMPLE_METHOD = {
    'gdp': Resampling.average,
    'pop': Resampling.sum,
}

TRAIN_RAW = {
    'gdp': r'READY_data/inputs/2019_gdp_aligned_010.tif',
    'pop': r'READY_data/inputs/2020_pop_aligned_010.tif',
}


def resample_only(input_path, kind):
    data = np.full((ref_height, ref_width), np.nan, dtype=np.float32)
    with rasterio.open(input_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=data,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=RESAMPLE_METHOD[kind],
        )
    for flag in [-3.4028235e+38, -3.402823e+38, -99999, -9999, -32768]:
        data[np.isclose(data, flag, rtol=1e-5)] = np.nan
    data[data < -1e10] = np.nan
    data[np.isinf(data)] = np.nan
    data[data < 0] = np.nan
    return data


def compute_training_stats(kind):
    """Compute log1p mean/std from the raw training raster."""
    print('Computing training stats for ' + kind.upper() + ' from: ' + TRAIN_RAW[kind])
    data = resample_only(TRAIN_RAW[kind], kind)
    valid = data[~np.isnan(data)]
    log_vals = np.log1p(valid)
    mean_val = float(log_vals.mean())
    std_val  = float(log_vals.std())
    max_val  = float(valid.max())
    print('  Training log1p mean=' + str(round(mean_val, 4)) + ', std=' + str(round(std_val, 4)) + ', raw_max=' + str(round(max_val, 2)))
    return mean_val, std_val, max_val


def compute_unit_scale(file_dict, kind, train_log1p_mean):
    """Compute a multiplicative scale factor to convert SSP values into training units.
    Derives the shared log1p mean across all SSP files, then returns
    exp(train_mean - ssp_mean) so that after scaling, log1p(x*scale) ~ log1p(x) - offset.
    This aligns the SSP distribution to the training distribution while preserving
    between-scenario ordering."""
    n = sum(len(v) for v in file_dict.values())
    print('Computing unit scale for ' + kind.upper() + ' across ' + str(n) + ' SSP files...')
    all_log_vals = []
    for ssp, years in file_dict.items():
        for year, path in years.items():
            d = resample_only(path, kind)
            valid = d[~np.isnan(d)]
            if len(valid):
                all_log_vals.append(np.log1p(valid))
    combined = np.concatenate(all_log_vals)
    ssp_log1p_mean = float(combined.mean())
    scale = float(np.exp(train_log1p_mean - ssp_log1p_mean))
    print('  SSP shared log1p mean=' + str(round(ssp_log1p_mean, 4)))
    print('  Train log1p mean     =' + str(round(train_log1p_mean, 4)))
    print('  Unit scale factor    =' + str(round(scale, 6)) + '  (SSP * scale -> training units)')
    return scale


def normalize_and_save(input_path, output_path, kind, train_mean, train_std, unit_scale=1.0, train_max=None):
    data = resample_only(input_path, kind)
    if unit_scale != 1.0:
        data = data * unit_scale
    if train_max is not None:
        data = np.where(~np.isnan(data), np.minimum(data, train_max), data)
    valid = ~np.isnan(data)
    if valid.sum() == 0:
        print('  WARNING: No valid data in ' + os.path.basename(input_path))
        return
    normalized = np.full_like(data, np.nan)
    normalized[valid] = (np.log1p(data[valid]) - train_mean) / (train_std + 1e-8)
    print('  ' + os.path.basename(input_path)
          + '  norm mean=' + str(round(float(np.nanmean(normalized)), 4))
          + '  range=[' + str(round(float(np.nanmin(normalized)), 3))
          + ', ' + str(round(float(np.nanmax(normalized)), 3)) + ']')
    profile = ref_profile.copy()
    profile.update(dtype='float32', count=1, nodata=np.nan)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(normalized[np.newaxis, :, :])


## 3. Process population files (15 files)

Runs the resample+normalize function on all 15 SSP population rasters (SSP1--"5 ---- 2030/2050/2100). Source files are at ~0.008° (~1 km). Output: `POP_SSP{n}_{year}_normalized.tif`. Skips files that already exist.

In [ ]:
POP_FILES = {
    'SSP1': {
        2030: r'READY_data\SSP1\SSP1_2030_EU_UK_POP_01.tif',
        2050: r'READY_data\SSP1\SSP1_2050_EU_UK_POP_01.tif',
        2100: r'READY_data\SSP1\SSP1_2100_EU_UK_POP_01.tif',
    },
    'SSP2': {
        2030: r'READY_data\SSP2\SSP2_2030_EU_UK_POP_01.tif',
        2050: r'READY_data\SSP2\SSP2_2050_EU_UK_POP_01.tif',
        2100: r'READY_data\SSP2\SSP2_2100_EU_UK_POP_01.tif',
    },
    'SSP3': {
        2030: r'READY_data\SSP3\SSP3_2030_EU_UK_POP_01.tif',
        2050: r'READY_data\SSP3\SSP3_2050_EU_UK_POP_01.tif',
        2100: r'READY_data\SSP3\SSP3_2100_EU_UK_POP_01.tif',
    },
    'SSP4': {
        2030: r'READY_data\SSP4\SSP4_2030_EU_UK_POP_01.tif',
        2050: r'READY_data\SSP4\SSP4_2050_EU_UK_POP_01.tif',
        2100: r'READY_data\SSP4\SSP4_2100_EU_UK_POP_01.tif',
    },
    'SSP5': {
        2030: r'READY_data\SSP5\SSP5_2030_EU_UK_POP_01.tif',
        2050: r'READY_data\SSP5\SSP5_2050_EU_UK_POP_01.tif',
        2100: r'READY_data\SSP5\SSP5_2100_EU_UK_POP_01.tif',
    },
}

pop_train_mean, pop_train_std, pop_train_max = load_training_stats('pop')
# POP: same source and units as training -> no unit conversion needed

print('Processing POP files...')
for ssp, years in POP_FILES.items():
    for year, in_path in years.items():
        out_path = os.path.join(OUTPUT_DIR, 'POP_' + ssp + '_' + str(year) + '_normalized.tif')
        normalize_and_save(in_path, out_path, 'pop', pop_train_mean, pop_train_std, train_max=pop_train_max)

print('All POP files processed.')


Computing training stats for POP from: READY_data/inputs/2020_pop_aligned_010.tif
  Training log1p mean=6.1638, std=3.0142, raw_max=3436316.25
Processing POP files...
  SSP1_2030_EU_UK_POP_01.tif  norm mean=0.1463  range=[-2.045, 2.782]
  SSP1_2050_EU_UK_POP_01.tif  norm mean=0.1095  range=[-2.045, 2.8]
  SSP1_2100_EU_UK_POP_01.tif  norm mean=-0.0125  range=[-2.045, 2.804]
  SSP2_2030_EU_UK_POP_01.tif  norm mean=0.1444  range=[-2.045, 2.781]
  SSP2_2050_EU_UK_POP_01.tif  norm mean=0.1059  range=[-2.045, 2.8]
  SSP2_2100_EU_UK_POP_01.tif  norm mean=0.0328  range=[-2.045, 2.828]
  SSP3_2030_EU_UK_POP_01.tif  norm mean=0.145  range=[-2.045, 2.781]
  SSP3_2050_EU_UK_POP_01.tif  norm mean=0.1132  range=[-2.045, 2.803]
  SSP3_2100_EU_UK_POP_01.tif  norm mean=0.0975  range=[-2.045, 2.896]
  SSP4_2030_EU_UK_POP_01.tif  norm mean=0.1412  range=[-2.045, 2.778]
  SSP4_2050_EU_UK_POP_01.tif  norm mean=0.0852  range=[-2.045, 2.78]
  SSP4_2100_EU_UK_POP_01.tif  norm mean=-0.0838  range=[-2.045, 2.77

## 4. Resolution check --" verify all outputs

Checks the resolution of raw inputs vs. normalized outputs. Confirms all 30 output files exist and are at 0.10°. Run this after processing to verify everything is correct before running `projections.ipynb`.

In [ ]:
_NORM_DIR = r'READY_data\inputs_normalized\ssp'
_SSPS  = ['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']
_YEARS = [2030, 2050, 2100]

# --- Raw input resolution check ---
print('=== RAW INPUT RESOLUTIONS ===')
raw_files = {
    'CISI label (target)':     r'READY_data\labels\2024_CISI_010deg_nearest.tif',
    'GDP training (2019)':     r'READY_data\inputs\2019_gdp_aligned_010.tif',
    'Pop training (2020)':     r'READY_data\inputs\2020_pop_aligned_010.tif',
    'Land cover (2020)':       r'READY_data\landuse_onehot\clipped_history_2020_onehot.tif',
    'GDP SSP raw (SSP1 2030)': r'READY_data\GDP SSP\GDP2030_ssp1.tif',
    'Pop SSP raw (SSP1 2030)': r'READY_data\SSP1\SSP1_2030_EU_UK_POP_01.tif',
}
print(f'  {"File":<35} {"Shape":>12} {"Res (°)":>10}  {"0.1° match?"}')
print('  ' + '-' * 75)
for label, path in raw_files.items():
    if not os.path.exists(path):
        print(f'  {label:<35} FILE NOT FOUND')
        continue
    with rasterio.open(path) as src:
        res_x = abs(src.transform[0])
        shape = f'{src.height}x{src.width}'
    ok = '--"' if abs(res_x - 0.1) < 0.001 else f'FAIL  ({res_x:.5f}° --" will be resampled)'
    print(f'  {label:<35} {shape:>12} {res_x:>10.5f}  {ok}')

# --- All 30 normalized output files ---
print('\n=== NORMALIZED OUTPUT FILES (all 30) ===')
print(f'  {"File":<40} {"Shape":>12} {"Res (°)":>10}  {"Status"}')
print('  ' + '-' * 80)

all_ok = True
for ssp in _SSPS:
    for year in _YEARS:
        for kind in ['POP', 'GDP']:
            fname = f'{kind}_{ssp}_{year}_normalized.tif'
            path  = os.path.join(_NORM_DIR, fname)
            if not os.path.exists(path):
                print(f'  {fname:<40} MISSING')
                all_ok = False
                continue
            with rasterio.open(path) as src:
                res_x = abs(src.transform[0])
                shape = f'{src.height}x{src.width}'
            ok = '--"' if abs(res_x - 0.1) < 0.001 else f'wrong res: {res_x:.5f}°'
            print(f'  {fname:<40} {shape:>12} {res_x:>10.5f}  {ok}')
            if abs(res_x - 0.1) >= 0.001:
                all_ok = False

print()
print('All 30 files present and at 0.1°.' if all_ok else 'WARNING: some files are missing or at wrong resolution.')

=== RAW INPUT RESOLUTIONS ===
  File                                       Shape    Res (°)  0.1° match?
  ---------------------------------------------------------------------------
  CISI label (target)                      485x570    0.10000  âœ“
  GDP training (2019)                      485x570    0.10000  âœ“
  Pop training (2020)                      485x570    0.10000  âœ“
  Land cover (2020)                        485x570    0.10000  âœ“
  GDP SSP raw (SSP1 2030)              18000x43200    0.00833  âœ—  (0.00833° â€” will be resampled)
  Pop SSP raw (SSP1 2030)                4554x8480    0.00833  âœ—  (0.00833° â€” will be resampled)

=== NORMALIZED OUTPUT FILES (all 30) ===
  File                                            Shape    Res (°)  Status
  --------------------------------------------------------------------------------
  POP_SSP1_2030_normalized.tif                  485x570    0.10000  âœ“
  GDP_SSP1_2030_normalized.tif                  485x570    0.10000  âœ“
  P

## 5. Process GDP files (15 files)

Same as step 3 but for GDP. Source files are in `READY_data/GDP SSP/` at ~0.008° resolution. Output: `GDP_SSP{n}_{year}_normalized.tif`. Skips files that already exist.

In [ ]:
def compute_unit_scale(file_dict, kind, train_log1p_mean):
    n = sum(len(v) for v in file_dict.values())
    print("Computing unit scale for " + kind.upper() + " across " + str(n) + " SSP files...")
    all_log_vals = []
    for ssp, years in file_dict.items():
        for year, path in years.items():
            d = resample_only(path, kind)
            valid = d[~np.isnan(d)]
            if len(valid):
                all_log_vals.append(np.log1p(valid))
    combined = np.concatenate(all_log_vals)
    ssp_log1p_mean = float(combined.mean())
    scale = float(np.exp(train_log1p_mean - ssp_log1p_mean))
    print("  SSP shared log1p mean=" + str(round(ssp_log1p_mean, 4)))
    print("  Train log1p mean     =" + str(round(train_log1p_mean, 4)))
    print("  Unit scale factor    =" + str(round(scale, 6)) + "  (SSP * scale -> training units)")
    return scale


GDP_FILES = {
    'SSP1': {
        2030: r'READY_data\GDP SSP\GDP2030_ssp1.tif',
        2050: r'READY_data\GDP SSP\GDP2050_ssp1.tif',
        2100: r'READY_data\GDP SSP\GDP2100_ssp1.tif',
    },
    'SSP2': {
        2030: r'READY_data\GDP SSP\GDP2030_ssp2.tif',
        2050: r'READY_data\GDP SSP\GDP2050_ssp2.tif',
        2100: r'READY_data\GDP SSP\GDP2100_ssp2.tif',
    },
    'SSP3': {
        2030: r'READY_data\GDP SSP\GDP2030_ssp3.tif',
        2050: r'READY_data\GDP SSP\GDP2050_ssp3.tif',
        2100: r'READY_data\GDP SSP\GDP2100_ssp3.tif',
    },
    'SSP4': {
        2030: r'READY_data\GDP SSP\GDP2030_ssp4.tif',
        2050: r'READY_data\GDP SSP\GDP2050_ssp4.tif',
        2100: r'READY_data\GDP SSP\GDP2100_ssp4.tif',
    },
    'SSP5': {
        2030: r'READY_data\GDP SSP\GDP2030_ssp5.tif',
        2050: r'READY_data\GDP SSP\GDP2050_ssp5.tif',
        2100: r'READY_data\GDP SSP\GDP2100_ssp5.tif',
    },
}

gdp_train_mean, gdp_train_std, gdp_train_max = load_training_stats('gdp')
# GDP: training (Chen & Gao 2021) and SSP (Wang & Sun 2022) are in different units.
# Compute a single multiplicative scale factor from the shared log1p mean of all 15 SSP files
# so that scaled SSP values fall in the same range as training values.
# Applying ONE scale to ALL files preserves between-scenario ordering.
gdp_unit_scale = compute_unit_scale(GDP_FILES, 'gdp', gdp_train_mean)

print('Processing GDP files...')
for ssp, years in GDP_FILES.items():
    for year, in_path in years.items():
        out_path = os.path.join(OUTPUT_DIR, 'GDP_' + ssp + '_' + str(year) + '_normalized.tif')
        normalize_and_save(in_path, out_path, 'gdp', gdp_train_mean, gdp_train_std, unit_scale=gdp_unit_scale, train_max=gdp_train_max)

print('All GDP files processed.')


Computing training stats for GDP from: READY_data/inputs/2019_gdp_aligned_010.tif
  Training log1p mean=0.7986, std=0.9003, raw_max=101.49
Computing unit scale for GDP across 15 SSP files...
  SSP shared log1p mean=3.4486
  Train log1p mean     =0.7986
  Unit scale factor    =0.070656  (SSP * scale -> training units)
Processing GDP files...
  GDP2030_ssp1.tif  norm mean=0.6731  range=[-0.887, 4.255]
  GDP2050_ssp1.tif  norm mean=0.6813  range=[-0.887, 4.255]
  GDP2100_ssp1.tif  norm mean=0.6904  range=[-0.887, 4.255]
  GDP2030_ssp2.tif  norm mean=0.674  range=[-0.887, 4.255]
  GDP2050_ssp2.tif  norm mean=0.6818  range=[-0.887, 4.255]
  GDP2100_ssp2.tif  norm mean=0.6953  range=[-0.887, 4.255]
  GDP2030_ssp3.tif  norm mean=0.6733  range=[-0.887, 4.255]
  GDP2050_ssp3.tif  norm mean=0.6762  range=[-0.887, 4.255]
  GDP2100_ssp3.tif  norm mean=0.6761  range=[-0.887, 4.255]
  GDP2030_ssp4.tif  norm mean=0.6733  range=[-0.887, 4.255]
  GDP2050_ssp4.tif  norm mean=0.6812  range=[-0.887, 4.255

In [ ]:
NORM_STATS = {
    'gdp': {
        'train_mean':  gdp_train_mean,
        'train_std':   gdp_train_std,
        'train_max':   gdp_train_max,
        'unit_scale':  gdp_unit_scale,
    },
    'pop': {
        'train_mean':  pop_train_mean,
        'train_std':   pop_train_std,
        'train_max':   pop_train_max,
        'unit_scale':  1.0,
    },
}

stats_path = r'READY_data/inputs_normalized/NORM_STATS_SSP.json'
with open(stats_path, 'w') as _f:
    json.dump(NORM_STATS, _f, indent=2)
print('Saved', stats_path)
print('GDP: train_mean=' + str(round(gdp_train_mean,4)) + '  train_std=' + str(round(gdp_train_std,4)) + '  unit_scale=' + str(round(gdp_unit_scale,6)))
print('POP: train_mean=' + str(round(pop_train_mean,4)) + '  train_std=' + str(round(pop_train_std,4)))


Saved READY_data/inputs_normalized/NORM_STATS_SSP.json
GDP: train_mean=0.7986  train_std=0.9003  unit_scale=0.070656
POP: train_mean=6.1638  train_std=3.0142


## 6. Verify GDP outputs

In [ ]:
_NORM_DIR = r'READY_data\inputs_normalized\ssp'
_SSPS  = ['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']
_YEARS = [2030, 2050, 2100]

print(f'{"File":<35} {"Shape":>12} {"Valid px":>10} {"Min":>8} {"Max":>8} {"Mean":>8}  {"OK?"}')
print('-' * 100)

all_ok = True
for ssp in _SSPS:
    for year in _YEARS:
        fname = f'GDP_{ssp}_{year}_normalized.tif'
        path  = os.path.join(_NORM_DIR, fname)
        if not os.path.exists(path):
            print(f'{fname:<35} MISSING')
            all_ok = False
            continue
        with rasterio.open(path) as src:
            data  = src.read(1).astype(np.float32)
            res_x = abs(src.transform[0])
            shape = f'{src.height}x{src.width}'
        valid = ~np.isnan(data)
        res_ok    = abs(res_x - 0.1) < 0.001
        has_data  = valid.sum() > 0
        no_extremes = np.all(np.abs(data[valid]) < 20) if has_data else False
        ok = 'OK' if (res_ok and has_data and no_extremes) else 'FAIL'
        if not (res_ok and has_data and no_extremes):
            all_ok = False
        print(f'{fname:<35} {shape:>12} {valid.sum():>10,} {np.nanmin(data):>8.3f} {np.nanmax(data):>8.3f} {np.nanmean(data):>8.4f}  {ok}')

print()
print('All GDP files OK.' if all_ok else 'WARNING: one or more GDP files have issues.')

File                                       Shape   Valid px      Min      Max     Mean  OK?
----------------------------------------------------------------------------------------------------
GDP_SSP1_2030_normalized.tif             485x570    276,450   -0.887    4.255   0.6731  ✓
GDP_SSP1_2050_normalized.tif             485x570    276,450   -0.887    4.255   0.6813  ✓
GDP_SSP1_2100_normalized.tif             485x570    276,450   -0.887    4.255   0.6904  ✓
GDP_SSP2_2030_normalized.tif             485x570    276,450   -0.887    4.255   0.6740  ✓
GDP_SSP2_2050_normalized.tif             485x570    276,450   -0.887    4.255   0.6818  ✓
GDP_SSP2_2100_normalized.tif             485x570    276,450   -0.887    4.255   0.6953  ✓
GDP_SSP3_2030_normalized.tif             485x570    276,450   -0.887    4.255   0.6733  ✓
GDP_SSP3_2050_normalized.tif             485x570    276,450   -0.887    4.255   0.6762  ✓
GDP_SSP3_2100_normalized.tif             485x570    276,450   -0.887    4.255   0.6761 

## 7. Process land cover files (15 scenarios x 7 classes = 105 files)

Reprojects global 1km LC rasters from EPSG:6933 to EPSG:4326, clips to the Europe 0.10° reference grid (-25 to 32° lon, 32.41 to 80.91° lat), one-hot encodes 7 classes, and saves one file per class per scenario.

Source: `Global 7-land-types LULC projection dataset under SSPs-RCPs (1)/`
Output: `LC_SSP{n}_{year}_class_{c}.tif` in `READY_data/inputs_normalized/ssp/`

In [ ]:
LC_BASE = r'Global 7-land-types LULC projection dataset under SSPs-RCPs'

LC_FILES = {
    'SSP1': {'rcp': 'RCP26', 'years': {2030: 'global_SSP1_RCP26_2030.tif', 2050: 'global_SSP1_RCP26_2050.tif', 2100: 'global_SSP1_RCP26_2100.tif'}},
    'SSP2': {'rcp': 'RCP45', 'years': {2030: 'global_SSP2_RCP45_2030.tif', 2050: 'global_SSP2_RCP45_2050.tif', 2100: 'global_SSP2_RCP45_2100.tif'}},
    'SSP3': {'rcp': 'RCP70', 'years': {2030: 'global_SSP3_RCP70_2030.tif', 2050: 'global_SSP3_RCP70_2050.tif', 2100: 'global_SSP3_RCP70_2100.tif'}},
    'SSP4': {'rcp': 'RCP60', 'years': {2030: 'global_SSP4_RCP60_2030.tif', 2050: 'global_SSP4_RCP60_2050.tif', 2100: 'global_SSP4_RCP60_2100.tif'}},
    'SSP5': {'rcp': 'RCP85', 'years': {2030: 'global_SSP5_RCP85_2030.tif', 2050: 'global_SSP5_RCP85_2050.tif', 2100: 'global_SSP5_RCP85_2100.tif'}},
}


def process_lc(in_path, out_dir, ssp, year):
    """Reproject, clip to Europe, one-hot encode, save 7 class files."""
    print(f'Processing LC: {os.path.basename(in_path)}')

    # Step 1: reproject to EPSG:4326 at 0.10 deg, clipped to Europe
    dst_crs = ref_crs
    dst_transform = ref_transform
    dst_h, dst_w = ref_height, ref_width

    data = np.zeros((dst_h, dst_w), dtype=np.uint8)

    with rasterio.open(in_path) as src:
        rio_reproject(
            source=rasterio.band(src, 1),
            destination=data,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=RioResampling.nearest,  # categorical data -- nearest neighbour
        )

    # Step 2: one-hot encode -- save one file per class (1-7)
    profile = ref_profile.copy()
    profile.update(dtype='float32', count=1, nodata=np.nan)

    for c in range(1, 8):
        out_path = os.path.join(out_dir, f'LC_{ssp}_{year}_class_{c}.tif')
        if os.path.exists(out_path):
            print(f'  Skipping class {c} (already exists)')
            continue
        class_mask = (data == c).astype(np.float32)
        # set pixels where data==0 (nodata/ocean) to NaN
        class_mask[data == 0] = np.nan
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(class_mask[np.newaxis, :, :])
        print(f'  Saved class {c} -> {out_path}')


for ssp, info in LC_FILES.items():
    rcp = info['rcp']
    folder = os.path.join(LC_BASE, f'{ssp}_{rcp}')
    for year, fname in info['years'].items():
        in_path = os.path.join(folder, fname)
        if not os.path.exists(in_path):
            print(f'MISSING: {in_path}')
            continue
        # skip if all 7 classes already done
        already = all(os.path.exists(os.path.join(OUTPUT_DIR, f'LC_{ssp}_{year}_class_{c}.tif')) for c in range(1,8))
        if already:
            print(f'Skipping {ssp} {year} LC (all 7 classes exist)')
            continue
        process_lc(in_path, OUTPUT_DIR, ssp, year)

print('All LC files processed.')

Skipping SSP1 2030 LC (all 7 classes exist)
Skipping SSP1 2050 LC (all 7 classes exist)
Skipping SSP1 2100 LC (all 7 classes exist)
Skipping SSP2 2030 LC (all 7 classes exist)
Skipping SSP2 2050 LC (all 7 classes exist)
Skipping SSP2 2100 LC (all 7 classes exist)
Skipping SSP3 2030 LC (all 7 classes exist)
Skipping SSP3 2050 LC (all 7 classes exist)
Skipping SSP3 2100 LC (all 7 classes exist)
Skipping SSP4 2030 LC (all 7 classes exist)
Skipping SSP4 2050 LC (all 7 classes exist)
Skipping SSP4 2100 LC (all 7 classes exist)
Skipping SSP5 2030 LC (all 7 classes exist)
Skipping SSP5 2050 LC (all 7 classes exist)
Skipping SSP5 2100 LC (all 7 classes exist)
All LC files processed.
